# LeetCode 2460 — Apply Operations to an Array bằng Python

Link đề: https://leetcode.com/problems/apply-operations-to-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def applyOperations(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **mô phỏng (simulation)** gồm hai giai đoạn: áp dụng các phép toán **tuần tự** `n - 1` lần, rồi **dồn mọi số `0` về cuối** mà giữ nguyên thứ tự các số khác.
2. Phân biệt **tuần tự** (phép toán sau nhìn thấy kết quả của phép toán trước) với **đồng thời** (dùng bản sao cũ) — chi tiết quyết định đáp án.
3. Dùng `sorted` với `key`, tính chất **ổn định (stable)** của sắp xếp, kỹ thuật **hai con trỏ ghi/đọc** để dồn phần tử tại chỗ.
4. Viết 4 cách: mô phỏng hai giai đoạn với danh sách mới, sắp xếp ổn định theo `x == 0`, hai con trỏ tại chỗ sau mô phỏng, và gộp hai giai đoạn trong một lượt; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: lọc số `0` **trước** khi thực hiện phép toán, dùng bản sao cũ để so sánh, sắp xếp giảm dần/tăng dần thay vì dồn số `0`, và duyệt thiếu một cặp cuối.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm các số nguyên không âm, đánh chỉ số từ 0, kích thước `n`. Bạn phải thực hiện `n - 1` phép toán **tuần tự**. Ở bước thứ `i` (`0 <= i <= n - 2`):

- Nếu `nums[i] == nums[i + 1]` thì **nhân đôi** `nums[i]` và **gán `0`** cho `nums[i + 1]`.
- Ngược lại, **bỏ qua** phép toán này.

Sau khi thực hiện **tất cả** phép toán, **dồn mọi số `0` về cuối** mảng (các số khác `0` giữ nguyên thứ tự tương đối). Trả về mảng kết quả.

Chú ý: phép toán áp dụng **tuần tự**, không phải đồng thời.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,2,1,1,0]` | `[1,4,2,0,0,0]` | `i=1`: `2==2` ⇒ `[1,4,0,1,1,0]`; `i=3`: `1==1` ⇒ `[1,4,0,2,0,0]`; dồn `0` ⇒ `[1,4,2,0,0,0]` |
| 2 | `[0,1]` | `[1,0]` | không phép toán nào; dồn `0` ra cuối |

Constraints: `2 <= nums.length <= 2000`; `0 <= nums[i] <= 1000`.

In [1]:
nums = [1, 2, 2, 1, 1, 0]
n = len(nums)
for i in range(n - 1):
    if nums[i] == nums[i + 1]:
        nums[i] *= 2
        nums[i + 1] = 0
    print(f"sau bước i={i}: {nums}")
print("trước khi dồn 0:", nums)
print("sau khi dồn 0:", [x for x in nums if x != 0] + [0] * nums.count(0))

sau bước i=0: [1, 2, 2, 1, 1, 0]
sau bước i=1: [1, 4, 0, 1, 1, 0]
sau bước i=2: [1, 4, 0, 1, 1, 0]
sau bước i=3: [1, 4, 0, 2, 0, 0]
sau bước i=4: [1, 4, 0, 2, 0, 0]
trước khi dồn 0: [1, 4, 0, 2, 0, 0]
sau khi dồn 0: [1, 4, 2, 0, 0, 0]


## 3. Kiến thức nền

### 3.1. Sửa danh sách tại chỗ và hệ quả "tuần tự"

Gán `nums[i] *= 2` và `nums[i + 1] = 0` **sửa trực tiếp** danh sách. Khi vòng lặp sang `i + 1`, ta so sánh `nums[i + 1]` (vừa đặt thành `0`) với `nums[i + 2]`. Vì vậy phép toán tuần tự **thấy** kết quả của phép toán trước.

In [2]:
nums = [2, 2, 2]
for i in range(len(nums) - 1):
    if nums[i] == nums[i + 1]:
        nums[i] *= 2
        nums[i + 1] = 0
    print(i, nums)

0 [4, 0, 2]
1 [4, 0, 2]


### 3.2. Sao chép danh sách: `nums[:]` hoặc `list(nums)`

Để không làm hỏng dữ liệu gốc, hãy làm việc trên bản sao `a = nums[:]` hoặc `a = list(nums)`. Gán `a = nums` **không** sao chép, chỉ thêm tên gọi cho cùng một danh sách.

In [3]:
nums = [1, 2, 3]
alias = nums
copy = nums[:]
alias[0] = 99
print(nums, copy)

[99, 2, 3] [1, 2, 3]


### 3.3. `sorted` với `key` và tính ổn định

`sorted(a, key=f)` sắp xếp theo `f(x)`. Python dùng thuật toán **ổn định**: các phần tử có cùng khóa **giữ nguyên thứ tự xuất hiện**. Với khóa `x == 0` (`False` = 0 đứng trước, `True` = 1 đứng sau), mọi số khác `0` đứng trước, mọi số `0` đứng sau và thứ tự các số khác `0` không đổi.

In [4]:
a = [1, 4, 0, 2, 0, 0]
print(sorted(a, key=lambda x: x == 0))
print(sorted(a, key=lambda x: x == 0) == [x for x in a if x != 0] + [x for x in a if x == 0])

[1, 4, 2, 0, 0, 0]
True


### 3.4. Hai con trỏ ghi/đọc (write/read pointers)

Dồn phần tử tại chỗ bằng hai chỉ số: `read` quét toàn mảng; `write` chỉ vị trí kế tiếp để ghi số khác `0`. Mỗi lần gặp `nums[read] != 0`, ghi `nums[write] = nums[read]`, tăng `write`. Cuối cùng điền `0` từ `write` đến hết.

In [5]:
a = [1, 4, 0, 2, 0, 0]
write = 0
for read in range(len(a)):
    if a[read] != 0:
        a[write] = a[read]
        write += 1
for k in range(write, len(a)):
    a[k] = 0
print(a)

[1, 4, 2, 0, 0, 0]


### 3.5. List comprehension và nhân danh sách

`[x for x in a if x != 0]` lọc số khác `0`; `[0] * k` tạo `k` số `0`. Ghép hai danh sách bằng `+`. `a.count(0)` đếm số lần xuất hiện của `0`.

In [6]:
a = [1, 4, 0, 2, 0, 0]
print([x for x in a if x != 0] + [0] * a.count(0))

[1, 4, 2, 0, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Chia thành hai việc độc lập:

**Việc A — áp dụng phép toán** (từ trái sang phải). Tại mỗi cặp liền kề bằng nhau, "gộp" hai số: bên trái nhân đôi, bên phải về `0`. Ví dụ `[1,2,2,1,1,0]`: cặp `(2,2)` gộp thành `4,0`; cặp `(1,1)` gộp thành `2,0`.

**Việc B — dồn số `0`**. Sau việc A ta có `[1,4,0,2,0,0]`. Bỏ các số `0` còn `[1,4,2]` rồi thêm ba số `0` ở cuối.

Vì sao **tuần tự** quan trọng? Với `[2,2,2]`: bước 0 làm `[4,0,2]`; bước 1 so `0` với `2` ⇒ khác, bỏ qua. Kết quả sau dồn `[4,2,0]`. Nếu làm "đồng thời" (so từng cặp trên mảng gốc) cả hai cặp đều bằng nhau và ta có kết quả khác.

In [7]:
nums = [2, 2, 2]
a = nums[:]
for i in range(len(a) - 1):
    if a[i] == a[i + 1]:
        a[i] *= 2
        a[i + 1] = 0
print("tuần tự:", a, "->", sorted(a, key=lambda x: x == 0))

tuần tự: [4, 0, 2] -> [4, 2, 0]


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng đề rồi dựng danh sách mới

**Ý tưởng:** sao chép `nums`; chạy `n - 1` bước theo đề; sau đó tạo danh sách mới gồm các số khác `0` (giữ thứ tự) rồi thêm các số `0`.

In [8]:
from typing import List

class SolutionBrute:
    def applyOperations(self, nums: List[int]) -> List[int]:
        a = nums[:]
        n = len(a)
        for i in range(n - 1):
            if a[i] == a[i + 1]:
                a[i] *= 2
                a[i + 1] = 0
        non_zero = [x for x in a if x != 0]
        return non_zero + [0] * (n - len(non_zero))

In [9]:
sol = SolutionBrute()
print(sol.applyOperations([1, 2, 2, 1, 1, 0]))   # [1, 4, 2, 0, 0, 0]
print(sol.applyOperations([0, 1]))               # [1, 0]
print(sol.applyOperations([2, 2, 2]))            # [4, 2, 0]

[1, 4, 2, 0, 0, 0]
[1, 0]
[4, 2, 0]


**Độ phức tạp:** `n = len(nums)`. Time `O(n)` (một lượt mô phỏng, một lượt lọc), Space `O(n)` cho bản sao và danh sách mới. **Nút thắt:** không có thắt cổ chai về thời gian; ta có thể giảm bộ nhớ phụ bằng cách dồn tại chỗ.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** hai giai đoạn **không cần phải tách biệt** về mặt chỉ số, và giai đoạn dồn có thể làm **tại chỗ** nhờ ổn định của thứ tự.

- **Invariant mô phỏng:** trước bước `i`, các vị trí `0..i-1` đã ở trạng thái **cuối cùng** (phép toán chỉ sửa `nums[i]` và `nums[i + 1]`, nên các vị trí trước `i` không bao giờ đổi nữa).
- **Hệ quả:** ngay sau bước `i`, `nums[i]` đã **chốt**. Nếu `nums[i] != 0`, có thể ghi ngay vào vị trí `write` (với `write <= i`) mà không ảnh hưởng đến các bước sau, vì các bước sau chỉ đọc `nums[i + 1]` trở đi.
- **Quy tắc:** nếu `nums[i] == nums[i + 1]` thì nhân đôi và đặt `nums[i + 1] = 0`; rồi **nếu** `nums[i] != 0` thì ghi vào `write` và tăng `write`.
- Cuối cùng xử lý phần tử cuối (`nums[n - 1]`, có thể vừa bị đặt `0`) rồi điền `0` đến hết.

## 7. Các cách tiếp theo

### Cách 2 — Mô phỏng + `sorted` ổn định theo `x == 0`

Giai đoạn dồn dùng khóa `x == 0`; ngắn gọn, nhờ tính ổn định mà đúng.

In [10]:
from typing import List

class SolutionSort:
    def applyOperations(self, nums: List[int]) -> List[int]:
        a = nums[:]
        for i in range(len(a) - 1):
            if a[i] == a[i + 1]:
                a[i] *= 2
                a[i + 1] = 0
        return sorted(a, key=lambda x: x == 0)

In [11]:
print(SolutionSort().applyOperations([1, 2, 2, 1, 1, 0]))   # [1, 4, 2, 0, 0, 0]
print(SolutionSort().applyOperations([0, 1]))               # [1, 0]
print(SolutionSort().applyOperations([2, 2, 2]))            # [4, 2, 0]

[1, 4, 2, 0, 0, 0]
[1, 0]
[4, 2, 0]


**Độ phức tạp:** `sorted` của Python (Timsort) là `O(n log n)` nhìn chung, nhưng với khóa chỉ có hai giá trị nên thực tế gần `O(n)`; để chắc chắn ta ghi `O(n log n)`. Space `O(n)`.

### Cách 3 — Mô phỏng + hai con trỏ ghi/đọc tại chỗ

Sau khi mô phỏng, dồn bằng `write`/`read`, không tạo danh sách phụ (ngoài bản sao để không sửa đầu vào).

In [12]:
from typing import List

class SolutionTwoPointers:
    def applyOperations(self, nums: List[int]) -> List[int]:
        a = nums[:]
        n = len(a)
        for i in range(n - 1):
            if a[i] == a[i + 1]:
                a[i] *= 2
                a[i + 1] = 0
        write = 0
        for read in range(n):
            if a[read] != 0:
                a[write] = a[read]
                write += 1
        for k in range(write, n):
            a[k] = 0
        return a

In [13]:
print(SolutionTwoPointers().applyOperations([1, 2, 2, 1, 1, 0]))   # [1, 4, 2, 0, 0, 0]
print(SolutionTwoPointers().applyOperations([0, 1]))               # [1, 0]
print(SolutionTwoPointers().applyOperations([2, 2, 2]))            # [4, 2, 0]

[1, 4, 2, 0, 0, 0]
[1, 0]
[4, 2, 0]


**Độ phức tạp:** Time `O(n)`, Space `O(1)` phụ (ngoài bản sao `O(n)`; nếu sửa trực tiếp `nums` thì `O(1)`).

### Cách 4 — Gộp hai giai đoạn trong một lượt

Dùng bất biến ở mục 6: sau khi quyết định xong `nums[i]`, ghi ngay vào `write`. Duyệt `i` từ `0` đến `n - 1`; ở mỗi `i < n - 1` thực hiện phép toán trước.

In [14]:
from typing import List

class SolutionOnePass:
    def applyOperations(self, nums: List[int]) -> List[int]:
        a = nums[:]
        n = len(a)
        write = 0
        for i in range(n):
            if i + 1 < n and a[i] == a[i + 1]:
                a[i] *= 2
                a[i + 1] = 0
            if a[i] != 0:
                a[write] = a[i]
                write += 1
        for k in range(write, n):
            a[k] = 0
        return a

In [15]:
print(SolutionOnePass().applyOperations([1, 2, 2, 1, 1, 0]))   # [1, 4, 2, 0, 0, 0]
print(SolutionOnePass().applyOperations([0, 1]))               # [1, 0]
print(SolutionOnePass().applyOperations([2, 2, 2]))            # [4, 2, 0]

[1, 4, 2, 0, 0, 0]
[1, 0]
[4, 2, 0]


**Độ phức tạp:** Time `O(n)` một lượt, Space `O(1)` phụ (ngoài bản sao). Điểm tinh tế: ta ghi vào `a[write]` với `write <= i`, nên không ghi đè phần tử chưa xử lý.

## 8. TRACE từng bước

Cách một lượt trên `[1,2,2,1,1,0]` (`write` là vị trí ghi kế tiếp):

| `i` | `a` trước | điều kiện `a[i]==a[i+1]` | `a` sau phép toán | `a[i] != 0`? | `write` sau | `a` sau ghi |
|---|---|---|---|---|---|---|
| 0 | `[1,2,2,1,1,0]` | `1==2` sai | không đổi | có | 1 | `[1,2,2,1,1,0]` |
| 1 | `[1,2,2,1,1,0]` | `2==2` đúng | `[1,4,0,1,1,0]` | có | 2 | `[1,4,0,1,1,0]` |
| 2 | `[1,4,0,1,1,0]` | `0==1` sai | không đổi | không | 2 | không ghi |
| 3 | `[1,4,0,1,1,0]` | `1==1` đúng | `[1,4,0,2,0,0]` | có | 3 | `[1,4,2,2,0,0]` (ghi `2` vào chỉ số 2) |
| 4 | `[1,4,2,2,0,0]` | `0==0` đúng | `a[4] = 0, a[5] = 0` | không | 3 | không ghi |
| 5 | `[1,4,2,2,0,0]` | (không có cặp) | không đổi | không | 3 | không ghi |

Cuối cùng điền `0` từ `write = 3` đến hết: `[1,4,2,0,0,0]`.

In [16]:
def trace_one_pass(nums):
    a = nums[:]
    n = len(a)
    write = 0
    for i in range(n):
        before = a[:]
        if i + 1 < n and a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
        wrote = False
        if a[i] != 0:
            a[write] = a[i]
            write += 1
            wrote = True
        print(f"i={i}: trước={before} -> sau={a}, write={write}, ghi={wrote}")
    for k in range(write, n):
        a[k] = 0
    print("kết quả:", a)
    return a

trace_one_pass([1, 2, 2, 1, 1, 0])
trace_one_pass([2, 2, 2])

i=0: trước=[1, 2, 2, 1, 1, 0] -> sau=[1, 2, 2, 1, 1, 0], write=1, ghi=True
i=1: trước=[1, 2, 2, 1, 1, 0] -> sau=[1, 4, 0, 1, 1, 0], write=2, ghi=True
i=2: trước=[1, 4, 0, 1, 1, 0] -> sau=[1, 4, 0, 1, 1, 0], write=2, ghi=False
i=3: trước=[1, 4, 0, 1, 1, 0] -> sau=[1, 4, 2, 2, 0, 0], write=3, ghi=True
i=4: trước=[1, 4, 2, 2, 0, 0] -> sau=[1, 4, 2, 2, 0, 0], write=3, ghi=False
i=5: trước=[1, 4, 2, 2, 0, 0] -> sau=[1, 4, 2, 2, 0, 0], write=3, ghi=False
kết quả: [1, 4, 2, 0, 0, 0]
i=0: trước=[2, 2, 2] -> sau=[4, 0, 2], write=1, ghi=True
i=1: trước=[4, 0, 2] -> sau=[4, 0, 2], write=1, ghi=False
i=2: trước=[4, 0, 2] -> sau=[4, 2, 2], write=2, ghi=True
kết quả: [4, 2, 0]


[4, 2, 0]

## 9. Code nộp LeetCode

Dùng cách hai giai đoạn gọn: mô phỏng rồi dồn tại chỗ (sửa trực tiếp `nums` được phép trên LeetCode):

In [17]:
from typing import List

class Solution:
    def applyOperations(self, nums: List[int]) -> List[int]:
        n = len(nums)
        for i in range(n - 1):
            if nums[i] == nums[i + 1]:
                nums[i] *= 2
                nums[i + 1] = 0
        write = 0
        for read in range(n):
            if nums[read] != 0:
                nums[write] = nums[read]
                write += 1
        for k in range(write, n):
            nums[k] = 0
        return nums

In [18]:
print(Solution().applyOperations([1, 2, 2, 1, 1, 0]))
print(Solution().applyOperations([0, 1]))

[1, 4, 2, 0, 0, 0]
[1, 0]


## 10. Vì sao thuật toán đúng

1. **Giai đoạn phép toán.** Vòng `for i in range(n - 1)` thực hiện đúng `n - 1` phép toán theo thứ tự; đọc/ghi trực tiếp trên mảng nên phép toán sau thấy kết quả phép toán trước (đúng "tuần tự" của đề).
2. **Phép toán chỉ ảnh hưởng `i` và `i + 1`.** Do đó sau bước `i`, giá trị `nums[i]` không bao giờ thay đổi nữa (bước `i + 1` chỉ chạm `i + 1` và `i + 2`). Đây là invariant quan trọng cho cách một lượt.
3. **Giai đoạn dồn.** Hai con trỏ giữ bất biến: `a[0..write-1]` là các số khác `0` trong `a[0..read-1]` **theo đúng thứ tự gốc**; vì `write <= read` nên không ghi đè phần tử chưa đọc. Sau vòng, `write` là số phần tử khác `0`; điền `0` cho phần còn lại.
4. **Sắp xếp ổn định.** Khóa `x == 0` chia thành hai nhóm: khác `0` (khóa `False`) đứng trước, `0` (khóa `True`) đứng sau; ổn định ⇒ thứ tự tương đối trong mỗi nhóm không đổi ⇒ trùng với kết quả dồn.
5. **Cách một lượt.** Ghi `a[i]` vào `a[write]` ngay sau khi chốt `a[i]` (`write <= i`): thứ tự các số khác `0` được bảo toàn và không phá phần tử chưa xử lý (`a[i + 1..]`).
6. Mọi cách cho cùng kết quả vì cùng thực hiện đúng chuỗi phép toán rồi bảo toàn thứ tự số khác `0`.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| Mô phỏng + danh sách mới | `O(n)` | `O(n)` |
| Mô phỏng + `sorted` ổn định | `O(n log n)` (thực tế gần `O(n)`) | `O(n)` |
| Mô phỏng + hai con trỏ | `O(n)` | `O(1)` phụ |
| Một lượt gộp | `O(n)` | `O(1)` phụ |

`n <= 2000` nên mọi cách đều rất nhanh. Cách hai con trỏ có bộ nhớ phụ thấp nhất và là mẫu quen thuộc (giống bài Move Zeroes).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionBrute(), SolutionSort(), SolutionTwoPointers(), SolutionOnePass(), Solution()]
cases = [
    ([1, 2, 2, 1, 1, 0], [1, 4, 2, 0, 0, 0]),   # ví dụ 1
    ([0, 1], [1, 0]),                           # ví dụ 2
    ([1, 1], [2, 0]),                           # n = 2, gộp
    ([0, 0], [0, 0]),                           # toàn 0: 0 == 0 => 0*2 = 0, 0
    ([2, 2, 2], [4, 2, 0]),                     # tuần tự: cặp thứ hai không còn bằng nhau
    ([1, 1, 1, 1], [2, 2, 0, 0]),
    ([1000, 1000], [2000, 0]),                  # vượt 1000 sau khi nhân đôi
    ([3, 3, 6, 6], [6, 12, 0, 0]),
    ([5, 0, 5], [5, 5, 0]),                     # số 0 ở giữa: hai số 5 không kề nhau lúc xét phép toán
    ([1, 2, 3], [1, 2, 3]),                     # không có phép toán nào
    ([4, 4, 0, 4], [8, 4, 0, 0]),
    ([0, 0, 1, 1], [2, 0, 0, 0]),
    ([7, 7, 7, 7, 7], [14, 14, 7, 0, 0]),       # 5 số 7: (0,1) gộp, (2,3) gộp, còn lại 7
]
for nums, want in cases:
    for s in sols:
        before = nums[:]
        got = s.applyOperations(nums[:])
        assert got == want, (type(s).__name__, nums, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 13


Kiểm tra `[7, 7, 7, 7, 7]`: `i=0`: `7==7` ⇒ `[14,0,7,7,7]`; `i=1`: `0` vs `7` bỏ; `i=2`: `7==7` ⇒ `[14,0,14,0,7]`; `i=3`: `0` vs `7` bỏ. Dồn: `[14,14,7,0,0]`. Khớp với bảng.

### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `nums` đúng constraints: `2 <= n <= 2000`, `0 <= nums[i] <= 1000`. Dùng bộ sinh có xác suất lặp cao (giá trị nhỏ) để thường xuyên có cặp bằng nhau. Ngoài đối chiếu giữa các cách, ta còn dùng một "oracle" mô phỏng thuần.

In [20]:
import random
random.seed(2460)

def oracle(nums):
    a = list(nums)
    for i in range(len(a) - 1):
        if a[i] == a[i + 1]:
            a[i] = a[i] * 2
            a[i + 1] = 0
    zeros = sum(1 for x in a if x == 0)
    return [x for x in a if x != 0] + [0] * zeros

for _ in range(1500):
    n = random.randint(2, 15)
    hi = random.choice([1, 2, 3, 10, 1000])
    nums = [random.randint(0, hi) for _ in range(n)]
    want = oracle(nums)
    for s in sols:
        copy = nums[:]
        assert s.applyOperations(copy) == want, (type(s).__name__, nums)
print("1500 test nhỏ: 5 cách khớp oracle")

for _ in range(40):
    n = random.randint(500, 2000)
    nums = [random.randint(0, random.choice([2, 5, 1000])) for _ in range(n)]
    want = oracle(nums)
    for s in sols:
        assert s.applyOperations(nums[:]) == want
print("40 test lớn khớp oracle")

1500 test nhỏ: 5 cách khớp oracle
40 test lớn khớp oracle


### 12.3. Tính chất của kết quả

- Độ dài không đổi.
- Số `0` ở cuối liền một khối (mọi số khác `0` đứng trước mọi số `0`).
- Tổng các phần tử không đổi: mỗi phép toán nhân đôi `x` và đặt `x` (số bằng nhau) thành `0`, nên `x + x = 2x` được bảo toàn.

In [21]:
random.seed(8)
for _ in range(500):
    n = random.randint(2, 30)
    nums = [random.randint(0, 5) for _ in range(n)]
    out = Solution().applyOperations(nums[:])
    assert len(out) == n
    k = out.index(0) if 0 in out else n
    assert all(x != 0 for x in out[:k]) and all(x == 0 for x in out[k:])
    assert sum(out) == sum(nums)
print("kích thước, khối số 0 ở cuối và tổng được bảo toàn")

kích thước, khối số 0 ở cuối và tổng được bảo toàn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lọc số `0` trước rồi mới thực hiện phép toán

In [22]:
from typing import List

def sai_loc_truoc(nums: List[int]) -> List[int]:
    a = [x for x in nums if x != 0]                           # SAI: bỏ 0 trước khi áp dụng phép toán
    for i in range(len(a) - 1):
        if a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
    non_zero = [x for x in a if x != 0]
    return non_zero + [0] * (len(nums) - len(non_zero))

nums = [5, 0, 5]
print("đúng:", Solution().applyOperations(nums[:]), "| sai:", sai_loc_truoc(nums))
assert Solution().applyOperations(nums[:]) != sai_loc_truoc(nums)

đúng: [5, 5, 0] | sai: [10, 0, 0]


**Vì sao sai:** phép toán chỉ xét hai phần tử **liền kề trong mảng gốc**. `[5, 0, 5]`: hai số `5` không kề nhau nên không gộp. Nếu lọc `0` trước, hai số `5` bỗng kề nhau và bị gộp thành `10`, sai. Đề yêu cầu: **làm phép toán trước, dồn `0` sau**.

### Lỗi 2: Dùng bản sao cũ để so sánh (làm "đồng thời")

In [23]:
def sai_dong_thoi(nums: List[int]) -> List[int]:
    original = nums[:]
    a = nums[:]
    for i in range(len(a) - 1):
        if original[i] == original[i + 1]:                    # SAI: so sánh trên bản gốc, không thấy cập nhật
            a[i] *= 2
            a[i + 1] = 0
    non_zero = [x for x in a if x != 0]
    return non_zero + [0] * (len(a) - len(non_zero))

nums = [2, 2, 2]
print("đúng:", Solution().applyOperations(nums[:]), "| sai:", sai_dong_thoi(nums))
assert Solution().applyOperations(nums[:]) != sai_dong_thoi(nums)

đúng: [4, 2, 0] | sai: [4, 0, 0]


**Vì sao sai:** phép toán phải **tuần tự**. Sau bước `0` ta có `[4, 0, 2]` nên bước `1` so `0` với `2` và bỏ qua. Dùng bản gốc `[2, 2, 2]` thì bước `1` lại thấy `2 == 2` và thực hiện thêm một phép toán.

### Lỗi 3: Dồn số `0` bằng `sort`

In [24]:
def sai_sort(nums: List[int]) -> List[int]:
    a = nums[:]
    for i in range(len(a) - 1):
        if a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
    a.sort(reverse=True)                                      # SAI: sắp xếp phá thứ tự tương đối
    return a

nums = [1, 2, 2, 1, 1, 0]
print("đúng:", Solution().applyOperations(nums[:]), "| sai:", sai_sort(nums))
assert Solution().applyOperations(nums[:]) != sai_sort(nums)

đúng: [1, 4, 2, 0, 0, 0] | sai: [4, 2, 1, 0, 0, 0]


**Vì sao sai:** đề yêu cầu **giữ nguyên thứ tự tương đối** của các số khác `0`. Sắp xếp giảm dần đưa `4` lên trước `1`. Cách đúng là sắp theo khóa `x == 0` (ổn định) hoặc dồn bằng hai con trỏ.

### Lỗi 4: Duyệt thiếu một cặp (`range(n - 2)`)

In [25]:
def sai_thieu_cap(nums: List[int]) -> List[int]:
    a = nums[:]
    for i in range(len(a) - 2):                               # SAI: phải là n - 1 bước
        if a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
    non_zero = [x for x in a if x != 0]
    return non_zero + [0] * (len(a) - len(non_zero))

nums = [1, 1]
print("đúng:", Solution().applyOperations(nums[:]), "| sai:", sai_thieu_cap(nums))
assert Solution().applyOperations(nums[:]) != sai_thieu_cap(nums)

đúng: [2, 0] | sai: [1, 1]


**Vì sao sai:** có đúng `n - 1` phép toán, `i` chạy `0..n-2` tức `range(n - 1)`. Với `n = 2`, `range(0)` không làm gì và cặp `(1, 1)` bị bỏ qua. Lỗi tương tự thường gặp khi so sánh `a[i]` với `a[i + 1]`: luôn kiểm tra `i + 1 < n`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề ghi "**tuần tự**" ⇒ làm việc trên chính mảng đang sửa; đừng so sánh trên bản sao cũ.
- Hai giai đoạn **độc lập**: làm phép toán trước, dồn `0` sau. Đừng đảo thứ tự.
- Dồn số `0` về cuối, giữ thứ tự ⇒ mẫu **Move Zeroes**: hai con trỏ `write`/`read` hoặc `sorted(key=lambda x: x == 0)`.
- Nhớ sửa trên bản sao nếu muốn giữ đầu vào nguyên vẹn (trên LeetCode sửa trực tiếp được phép).
- Hãy nhắm vào các test đặc biệt: `[1,1]`, `[2,2,2]`, `[5,0,5]`, `[0,0]` và chuỗi toàn số giống nhau.
- Trước khi nộp, viết một hàm mô phỏng thuần để đối chiếu ngẫu nhiên (như `oracle` trong notebook).

## 15. Biến thể của bài

1. **Move Zeroes (LeetCode 283):** chỉ riêng giai đoạn dồn `0`.
2. **Gộp các cặp bằng nhau theo kiểu game 2048:** gộp theo một hướng, tính điểm.
3. **Xóa các số `0` thay vì dồn** (kết quả ngắn hơn).
4. **Phép toán khác:** ví dụ cộng hai số kề nhau khi bằng nhau, đặt số bên phải bằng `0`.
5. **Thực hiện phép toán từ phải sang trái**: kết quả sẽ khác với `[2,2,2]`.
6. **Dồn số `0` về đầu** thay vì cuối.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng + danh sách mới | lọc số khác `0`, thêm `0` | `O(n)` | `O(n)` | rõ ràng nhất |
| 2. `sorted` ổn định | khóa `x == 0` | `O(n log n)` | `O(n)` | code ngắn, nhờ tính ổn định |
| 3. Hai con trỏ | `write`/`read` tại chỗ | `O(n)` | `O(1)` phụ | mẫu phỏng vấn kinh điển |
| 4. Một lượt gộp | chốt `a[i]` rồi ghi ngay vào `write` | `O(n)` | `O(1)` phụ | tối ưu, cần hiểu invariant |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().applyOperations([3, 3, 0, 0])` là gì?

**Bài 2 (tính tay).** Với `nums = [2, 2, 2, 2]`, viết mảng sau từng bước `i = 0, 1, 2` rồi sau khi dồn `0`.

**Bài 3 (điền TODO).** Hoàn thành hàm `don_so_0(a)` dồn số `0` về cuối, giữ thứ tự, tại chỗ.

In [26]:
def don_so_0(a):
    write = 0
    for read in range(len(a)):
        # TODO: nếu a[read] khác 0 thì ghi vào a[write] và tăng write
        pass
    # TODO: điền 0 từ write đến hết
    return a

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `apply_right_to_left(nums)`: thực hiện phép toán theo thứ tự `i = n - 2, n - 3, ..., 0` (từ phải sang trái), rồi dồn `0`. So sánh với đáp án chính trên `[2, 2, 2]`.

**Bài 5 (biến thể / mini challenge).** Viết `merge_2048(nums)`: thực hiện đúng quy tắc của đề nhưng sau khi gộp một cặp thì **bỏ qua** phần tử kế tiếp (không dùng `0` đã đặt), ví dụ `[2,2,2,2]` → `[4,4,0,0]`, `[2,2,2]` → `[4,2,0]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3 == 3` ⇒ `[6, 0, 0, 0]`; sau đó `0 == 0` thay bằng `0`; kết quả vẫn `[6, 0, 0, 0]`. Bài 2: bước 0: `[4,0,2,2]`; bước 1: `0` vs `2` bỏ; bước 2: `2==2` ⇒ `[4,0,4,0]`.
- **Mức 2 (vừa):** Bài 3: `a[write] = a[read]; write += 1` khi `a[read] != 0`; sau vòng lặp `for k in range(write, len(a)): a[k] = 0`. Bài 4: `for i in range(n - 2, -1, -1)`.
- **Mức 3 (sâu):** Bài 5: dùng `i` tăng thủ công: nếu `a[i] == a[i + 1]` thì gộp rồi `i += 2`; ngược lại `i += 1`. Thực ra kết quả luôn trùng với đề chính: sau khi gộp, `a[i + 1] = 0`, phép toán kế tiếp so `0` với `a[i + 2]`; nếu `a[i + 2] != 0` thì bỏ qua, nếu bằng `0` thì `0 * 2 = 0` và `0` không đổi gì. Hãy tự kiểm tra bằng test ngẫu nhiên.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert Solution().applyOperations([3, 3, 0, 0]) == [6, 0, 0, 0]

# Bài 2: [2,2,2,2] -> [4,0,2,2] -> [4,0,2,2] -> [4,0,4,0] -> dồn: [4,4,0,0]
assert Solution().applyOperations([2, 2, 2, 2]) == [4, 4, 0, 0]

# Bài 3
def don_so_0(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    for k in range(write, len(a)):
        a[k] = 0
    return a

# Bài 4
def apply_right_to_left(nums):
    a = nums[:]
    for i in range(len(a) - 2, -1, -1):
        if a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
    return don_so_0(a)

# Bài 5
def merge_2048(nums):
    a = nums[:]
    i = 0
    while i < len(a) - 1:
        if a[i] == a[i + 1]:
            a[i] *= 2
            a[i + 1] = 0
            i += 2
        else:
            i += 1
    return don_so_0(a)

assert don_so_0([0, 1, 0, 2]) == [1, 2, 0, 0]
assert don_so_0([0, 0]) == [0, 0]
assert don_so_0([1, 2, 3]) == [1, 2, 3]
assert apply_right_to_left([2, 2, 2]) == [2, 4, 0]       # i=1: [2,4,0]; i=0: 2 vs 4 bỏ
assert apply_right_to_left([1, 1]) == [2, 0]
assert merge_2048([2, 2, 2, 2]) == [4, 4, 0, 0]
assert merge_2048([2, 2, 2]) == [4, 2, 0]
assert merge_2048([1, 1, 1]) == [2, 1, 0]
assert merge_2048([1, 1, 0, 0]) == Solution().applyOperations([1, 1, 0, 0]) == [2, 0, 0, 0]
# merge_2048 luôn trùng với đề chính
import random
random.seed(1)
for _ in range(300):
    arr = [random.randint(0, 3) for _ in range(random.randint(2, 12))]
    assert merge_2048(arr) == Solution().applyOperations(arr[:])
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (phép toán tuần tự rồi dồn `0` bằng hai con trỏ)

```python
class Solution:
    def applyOperations(self, nums: List[int]) -> List[int]:
        n = len(nums)
        for i in range(n - 1):
            if nums[i] == nums[i + 1]:
                nums[i] *= 2
                nums[i + 1] = 0
        write = 0
        for read in range(n):
            if nums[read] != 0:
                nums[write] = nums[read]
                write += 1
        for k in range(write, n):
            nums[k] = 0
        return nums
```

**Checklist**

- [ ] Chạy đúng `n - 1` phép toán (`range(n - 1)`), **trước** khi dồn số `0`?
- [ ] Phép toán dùng chính mảng đang sửa (tuần tự), không dùng bản sao cũ để so sánh?
- [ ] Dồn `0` **giữ nguyên thứ tự** các số khác `0` (không dùng `sort()` thường)?
- [ ] Hai con trỏ: `write <= read` và điền `0` từ `write` đến hết?
- [ ] Đã thử `[1,1]`, `[2,2,2]`, `[5,0,5]`, `[0,0]`?
- [ ] Giá trị sau khi nhân đôi có thể vượt `1000` (không giới hạn trong Python)?
- [ ] Signature đúng `applyOperations(self, nums: List[int]) -> List[int]` và có `from typing import List`?